In [ ]:
!pip install rasterio contextily

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 43.3 MB/s eta 0:00:00


In [ ]:
import contextily as ctx
import matplotlib.pyplot as plt
import numpy as np
import ee
import geemap
import json
from datetime import datetime
import requests
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import zipfile
import rasterio
from PIL import Image
import io
from google.colab import files

# Cập nhật danh sách tỉnh ĐBSCL
MEKONG_PROVINCES = [
    'An Giang', 'Ben Tre', 'Ca Mau', 'Can Tho city', 'Dong Thap',
    'Hau Giang', 'Kien Giang', 'Long An', 'Soc Trang',
    'Tien Giang', 'Tra Vinh', 'Vinh Long', 'Bac Lieu'
]

# Xác thực và khởi tạo Earth Engine
try:
    ee.Initialize(project='ee-python-api-471906')
    print("Earth Engine initialized successfully")
except Exception as e:
    print(f"Failed to initialize Earth Engine: {e}")
    ee.Authenticate()
    ee.Initialize(project='ee-python-api-471906')

def get_mekong_region():
    """Lấy geometry của vùng ĐBSCL"""
    try:
        provinces = ee.FeatureCollection("FAO/GAUL/2015/level1") \
            .filter(ee.Filter.eq('ADM0_NAME', 'Viet Nam'))
        mekong_fc = provinces.filter(ee.Filter.inList('ADM1_NAME', MEKONG_PROVINCES))
        return mekong_fc.union().geometry()
    except Exception as e:
        print(f"Error getting Mekong region: {e}")
        raise

def get_s1_collection(region, start_date, end_date):
    """Lấy bộ sưu tập dữ liệu Sentinel-1"""
    try:
        collection = (ee.ImageCollection("COPERNICUS/S1_GRD")
                      .filterBounds(region)
                      .filterDate(start_date, end_date)
                      # Thêm các bộ lọc cụ thể cho Sentinel-1
                      .filter(ee.Filter.eq('instrumentMode', 'IW'))
                      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                      .filter(ee.Filter.eq('resolution_meters', 10))
                      .sort('system:time_start'))

        size = collection.size().getInfo()
        if size == 0:
            print(f"No Sentinel-1 images found for {start_date} to {end_date}")
            return None
        print(f"Found {size} Sentinel-1 images")
        return collection
    except Exception as e:
        print(f"Error getting Sentinel-1 collection: {e}")
        return None

def download_image_directly(image, output_subdir, scale=100, bands=['VV', 'VH']):
    """Tải ảnh Sentinel-1 trực tiếp về local qua EE API"""
    try:
        image_id = image.get('system:index').getInfo()
        url = image.select(bands).getDownloadURL({
            'region': image.geometry(),
            'scale': scale,
            'format': 'GEO_TIFF',
            'crs': 'EPSG:4326',
            'filePerBand': False
        })

        response = requests.get(url, timeout=60)
        if response.status_code == 200:
            content_type = response.headers.get('content-type', '')
            if 'zip' in content_type:
                filename = f"{image_id.split('/')[-1]}.zip"
                filepath = os.path.join(output_subdir, filename)
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                with zipfile.ZipFile(filepath, 'r') as zip_ref:
                    zip_ref.extractall(output_subdir)
                os.remove(filepath)
                # Tên file tif sau khi giải nén có thể khác, cần tìm file
                tif_files = [f for f in os.listdir(output_subdir) if f.endswith('.tif')]
                return os.path.join(output_subdir, tif_files[0]) if tif_files else None
            else:
                filename = f"{image_id.split('/')[-1]}.tif"
                filepath = os.path.join(output_subdir, filename)
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                return filepath
        else:
            return None
    except Exception as e:
        return None

def download_batch_images(collection, region, output_dir, max_images=None, scale=100):
    """Tải hàng loạt ảnh Sentinel-1 với multi-threading"""
    if max_images == None:
        images_list = collection.toList(collection.size())
        total_images = images_list.size().getInfo()
    else:
        images_list = collection.toList(max_images)
        total_images = min(max_images, images_list.size().getInfo())

    results = []
    with tqdm(total=total_images, desc="Tải ảnh Sentinel-1", unit="ảnh") as pbar:
        for i in range(total_images):
            try:
                img = ee.Image(images_list.get(i)).clip(region)
                image_date_millis = img.get('system:time_start').getInfo()
                date_time_str = datetime.fromtimestamp(image_date_millis / 1000).strftime('%Y-%m-%d_%H-%M-%S')
                image_subdir = os.path.join(output_dir, f"S1_{date_time_str}")
                os.makedirs(image_subdir, exist_ok=True)
                pbar.set_postfix_str(f"Processing: S1_{date_time_str}")
                result = download_image_directly(img, image_subdir, scale)
                save_image_metadata(img, image_subdir)
                create_matplotlib_quicklook_s1(img, region, image_subdir)
                if result:
                    results.append(result)
                time.sleep(2)
                pbar.update(1)
            except Exception as e:
                pbar.write(f"❌ Lỗi xử lý ảnh {i+1}: {e}")
    return results

def save_image_metadata(image, output_subdir):
    """Lưu metadata của ảnh"""
    try:
        image_id = image.get('system:index').getInfo()
        properties = image.toDictionary().getInfo()
        metadata_filename = f"{image_id.split('/')[-1]}_metadata.json"
        metadata_path = os.path.join(output_subdir, metadata_filename)
        with open(metadata_path, 'w', encoding='utf-8') as f:
            json.dump(properties, f, ensure_ascii=False, indent=4)
        return metadata_path
    except Exception as e:
        return None

def create_matplotlib_quicklook_s1(image, region, output_subdir, scale=200):
    """
    Tạo ảnh quicklook màu giả bằng Matplotlib cho Sentinel-1 (VV, VH)
    """
    try:
        image_id = image.get('system:index').getInfo()
        image_date_millis = image.get('system:time_start').getInfo()

        # Trực quan hóa Sentinel-1 với pseudocolor
        # Ví dụ: kết hợp VV và VH
        vv_band = image.select('VV').visualize(min=-25, max=0)
        vh_band = image.select('VH').visualize(min=-25, max=0)

        # Tạo ảnh giả màu từ 2 băng tần
        rgb_img_vis = ee.Image.cat([vh_band, vv_band, vh_band]).visualize()

        rgb_array = geemap.ee_to_numpy(
            rgb_img_vis,
            region=region,
            scale=scale
        )

        # Xử lý các giá trị NaN và 0 nếu có
        rgb_array = np.nan_to_num(rgb_array, nan=255)
        rgb_array[rgb_array == 0] = 255


        image_date_title = datetime.fromtimestamp(image_date_millis / 1000).strftime('%d-%m-%Y %H:%M:%S')
        bounds = region.bounds().getInfo()["coordinates"][0]
        minx = min([c[0] for c in bounds])
        maxx = max([c[0] for c in bounds])
        miny = min([c[1] for c in bounds])
        maxy = max([c[1] for c in bounds])
        if miny > maxy:
            miny, maxy = maxy, miny

        fig, ax = plt.subplots(figsize=(12, 12))
        ax.set_aspect('equal')
        ax.imshow(rgb_array, extent=[minx, maxx, miny, maxy])

        mekong_boundary_coords = region.getInfo()['coordinates']
        for polygon in mekong_boundary_coords:
            for ring in polygon:
                lon = [point[0] for point in ring]
                lat = [point[1] for point in ring]
                ax.plot(lon, lat, color='black', linewidth=1.5, alpha=0.8)

        ax.set_title(f'Ảnh vệ tinh Sentinel-1 ĐBSCL ({image_date_title})', fontsize=16)
        ax.set_xlabel('Kinh độ', fontsize=12)
        ax.set_ylabel('Vĩ độ', fontsize=12)
        ax.autoscale(enable=True, axis='both', tight=True)
        plt.grid(True, linestyle='--', alpha=0.6)

        filename = f"{image_id.split('/')[-1]}_preview_s1.png"
        filepath = os.path.join(output_subdir, filename)
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close(fig)
        return filepath
    except Exception as e:
        return None

def main():
    year = 2019
    months = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
    mekong_region = get_mekong_region()

    for idx in range(len(months)):
        OUTPUT_DIR = f"/content/dbscl-sentinel-1_{year}-{months[idx]}"
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        start_date = f'{year}-{months[idx]}-01'
        if idx == len(months) - 1:
            end_date = f'{year}-12-31' # Sửa lại để tải hết tháng cuối cùng
        else:
            end_date = f'{year}-{months[idx+1]}-01'

        # Sửa tên hàm get_s1_collection
        collection = get_s1_collection(mekong_region, start_date, end_date)

        if collection is None:
            print(f"Không tìm thấy ảnh Sentinel-1 cho tháng {months[idx]}. Bỏ qua...")
            continue

        print(f"🚀 Bắt đầu tải ảnh vệ tinh Sentinel-1 cho tháng {months[idx]}/{year}...")
        downloaded_files = download_batch_images(
            collection, mekong_region, OUTPUT_DIR, scale=100
        )
        print(f"✅ Đã hoàn tất xử lý {len(downloaded_files)} ảnh trong tháng.")

    print("📦 Đang nén tất cả các folder của các tháng thành một file zip duy nhất...")
    all_months_zip = f'/content/dbscl-sentinel-1_{year}.zip'
    os.system(f'zip -r {all_months_zip} /content/dbscl-sentinel-1_{year}*')

    print("⬇️ Bắt đầu tải file cuối cùng...")
    files.download(all_months_zip)

if __name__ == "__main__":
    main()

Earth Engine initialized successfully
Found 42 Sentinel-1 images
🚀 Bắt đầu tải ảnh vệ tinh Sentinel-1 cho tháng 01/2019...


Tải ảnh Sentinel-1: 100%|██████████| 42/42 [13:23<00:00, 19.13s/ảnh, Processing: S1_2019-01-31_22-45-20]


✅ Đã hoàn tất xử lý 30 ảnh trong tháng.
Found 33 Sentinel-1 images
🚀 Bắt đầu tải ảnh vệ tinh Sentinel-1 cho tháng 02/2019...


Tải ảnh Sentinel-1: 100%|██████████| 33/33 [10:28<00:00, 19.06s/ảnh, Processing: S1_2019-02-26_11-03-07]


✅ Đã hoàn tất xử lý 25 ảnh trong tháng.
Found 36 Sentinel-1 images
🚀 Bắt đầu tải ảnh vệ tinh Sentinel-1 cho tháng 03/2019...


...

✅ Đã hoàn tất xử lý 28 ảnh trong tháng.
Found 33 Sentinel-1 images
🚀 Bắt đầu tải ảnh vệ tinh Sentinel-1 cho tháng 12/2019...


Tải ảnh Sentinel-1: 100%|██████████| 33/33 [10:02<00:00, 18.27s/ảnh, Processing: S1_2019-12-28_11-11-29]


✅ Đã hoàn tất xử lý 22 ảnh trong tháng.
📦 Đang nén tất cả các folder của các tháng thành một file zip duy nhất...
⬇️ Bắt đầu tải file cuối cùng...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>